# Company Brochure Generator with Gradio

This notebook builds an interactive web app that generates a company brochure from a given landing page URL. It scrapes the website content, sends it to a Gemini LLM, and displays the result through a Gradio interface.

## Imports

We import the OpenAI client (used here as a wrapper for the Gemini API), Gradio for the UI, and our custom `fetch_website_contents` function for scraping.

In [4]:
import os
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv
import gradio as gr
from my_scraper import fetch_website_contents

## API Configuration

Load the API key from the `.env` file and set up the OpenAI-compatible client pointing to Google's Gemini API endpoint.

In [3]:
load_dotenv(override=True)
API_KEY = os.getenv('GOOGLE_API_KEY')
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = 'gemini-2.5-flash'

gemini = OpenAI(api_key=API_KEY, base_url=BASE_URL)

## System Prompt

This message tells the model what role it should play. It instructs the LLM to analyze a company's landing page and produce a short brochure aimed at customers, investors, and recruits.

In [5]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

## Streaming Response Function

This function sends the prompt to the Gemini model and yields the response incrementally. Using streaming lets the user see the brochure being generated in real time instead of waiting for the full response.

In [9]:
def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

## Brochure Generation Pipeline

This function ties everything together. It takes a company name and URL, scrapes the landing page content using `fetch_website_contents`, builds the prompt, and streams the generated brochure back to the caller.

In [7]:
def stream_brochure(company_name, url):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    result = stream_gemini(prompt)
    yield from result

## Gradio Interface

Finally, we create the Gradio UI. The user provides a company name and a landing page URL, and the app displays the generated brochure. A couple of example inputs are included for quick testing.

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.com"],
            ["Anthropic", "https://anthropic.com"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


[2026-03-06 19:20:18] INFO: Fetched (200) <GET https://www.anthropic.com/> (referer: https://www.google.com/search?q=anthropic)
[2026-03-06 19:20:39] INFO: Fetched (200) <GET https://huggingface.co/> (referer: https://www.google.com/search?q=huggingface)
